# 🏦 SGCI — Framework DQ Simplifié
### Contrôles essentiels pour toutes tes extractions Teradata
---
**Ce que fait ce notebook :**
Pour chaque requête SQL que tu exécutes, il vérifie automatiquement :

| Contrôle | Question posée |
|---|---|
| Unicité | Y a-t-il des lignes entièrement dupliquées ? |
| Complétude | Le nombre de lignes est-il dans l'intervalle attendu ? |
| Fraîcheur | Les données contiennent-elles bien des lignes dans la période d'extraction ? |
| Cohérence | Aucune colonne n'est-elle entièrement vide ? |
| Validité | Les colonnes sont-elles dans le bon type ? |


## 0. Setup — imports, logger, connexion

In [ ]:
import pandas as pd
import numpy as np
import logging
import os
import time
import warnings
from datetime import datetime, date, timedelta
from pathlib import Path
import pytz
warnings.filterwarnings('ignore')

# ── Constantes ────────────────────────────────────────────────────────────────
LOG_DIR       = "./JOURNAL/"
LOG_FILE_NAME = "journal_dq.csv"
TZ_ABIDJAN    = pytz.timezone("Africa/Abidjan")
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

current_directory = os.getcwd()
USER = current_directory.split("/")[2] if len(current_directory.split("/")) > 2 else "sgci_user"

# ── Logger ────────────────────────────────────────────────────────────────────
def setup_logger(project_name: str, log_dir: str) -> logging.Logger:
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(project_name)
    logger.setLevel(logging.DEBUG)
    if logger.handlers:
        logger.handlers.clear()
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    fh = logging.FileHandler(f"{log_dir}/{project_name}.log", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    logger.propagate = False
    return logger

logger = setup_logger("sgci_dq", LOG_DIR)
print(f"✅ Setup OK — Utilisateur : {USER}")


In [ ]:
# ── Connexion Teradata ────────────────────────────────────────────────────────
def get_connection():
    """
    Connexion sécurisée à Teradata via variables d'environnement.
    Définir avant usage :
        os.environ["TD_HOST"]     = "..."
        os.environ["TD_USER"]     = "..."
        os.environ["TD_PASSWORD"] = "..."
    """
    import teradatasql
    return teradatasql.connect(
        host     = os.getenv("TD_HOST"),
        user     = os.getenv("TD_USER"),
        password = os.getenv("TD_PASSWORD"),
        logmech  = "LDAP"
    )

def extraire(query: str, nom: str) -> pd.DataFrame:
    """Exécute une requête Teradata et retourne un DataFrame."""
    logger.info(f"[{nom}] Extraction en cours...")
    t0 = time.time()
    with get_connection() as con:
        df = pd.read_sql(query, con)
    logger.info(f"[{nom}] {len(df):,} lignes extraites en {round(time.time()-t0,2)}s")
    return df

print("✅ Fonctions de connexion définies")
print("   → En production : décommenter get_connection() et appeler extraire()")


## 1. Les 5 contrôles DQ simplifiés

> Chaque contrôle est une fonction indépendante, simple et lisible.
> Toutes retournent le même format : `{"statut": "OK"/"ECHEC", "message": "..."}`.


In [ ]:
# ════════════════════════════════════════════════════════════
#  CONTRÔLE 1 — UNICITÉ
#  Vérifie s'il y a des lignes entièrement dupliquées.
#  Une ligne dupliquée = toutes ses colonnes sont identiques
#  à une autre ligne du DataFrame.
# ════════════════════════════════════════════════════════════
def check_unicite(df: pd.DataFrame) -> dict:
    nb_doublons = df.duplicated(keep=False).sum()
    if nb_doublons > 0:
        return {
            "statut"  : "ECHEC",
            "message" : f"{nb_doublons} ligne(s) entièrement dupliquée(s) détectée(s)"
        }
    return {"statut": "OK", "message": "Aucun doublon détecté"}


# ════════════════════════════════════════════════════════════
#  CONTRÔLE 2 — COMPLÉTUDE
#  Vérifie que le nombre de lignes extraites est dans
#  l'intervalle [min_lignes, max_lignes] que tu définis.
#  Exemple : tu attends entre 100 et 5000 lignes par jour.
# ════════════════════════════════════════════════════════════
def check_completude(df: pd.DataFrame, min_lignes: int, max_lignes: int) -> dict:
    n = len(df)
    if n < min_lignes:
        return {
            "statut"  : "ECHEC",
            "message" : f"{n} ligne(s) extraite(s) — en dessous du minimum attendu ({min_lignes})"
        }
    if n > max_lignes:
        return {
            "statut"  : "ECHEC",
            "message" : f"{n} ligne(s) extraite(s) — au-dessus du maximum attendu ({max_lignes})"
        }
    return {"statut": "OK", "message": f"{n} ligne(s) dans l'intervalle [{min_lignes}, {max_lignes}]"}


# ════════════════════════════════════════════════════════════
#  CONTRÔLE 3 — FRAÎCHEUR
#  Vérifie que le DataFrame contient bien des lignes dans
#  la période d'extraction définie (date_debut, date_fin).
#  Si aucune ligne ne tombe dans cette période → ECHEC.
# ════════════════════════════════════════════════════════════
def check_fraicheur(df: pd.DataFrame, col_date: str,
                    date_debut: date, date_fin: date) -> dict:
    if col_date not in df.columns:
        return {
            "statut"  : "ECHEC",
            "message" : f"Colonne '{col_date}' absente du DataFrame"
        }
    dates = pd.to_datetime(df[col_date], errors="coerce").dt.date
    dans_periode = dates.between(date_debut, date_fin).sum()
    if dans_periode == 0:
        return {
            "statut"  : "ECHEC",
            "message" : (f"Aucune ligne dans la période [{date_debut}, {date_fin}]. "
                         f"Dates trouvées : {dates.min()} → {dates.max()}")
        }
    return {
        "statut"  : "OK",
        "message" : f"{dans_periode} ligne(s) dans la période [{date_debut}, {date_fin}]"
    }


# ════════════════════════════════════════════════════════════
#  CONTRÔLE 4 — COHÉRENCE
#  Vérifie qu'aucune colonne du DataFrame n'est entièrement
#  vide (100% de valeurs nulles).
#  Une colonne entièrement vide = problème d'extraction SQL.
# ════════════════════════════════════════════════════════════
def check_coherence(df: pd.DataFrame) -> dict:
    colonnes_vides = [
        col for col in df.columns
        if df[col].isnull().all()
    ]
    if colonnes_vides:
        return {
            "statut"  : "ECHEC",
            "message" : f"Colonne(s) entièrement vide(s) : {colonnes_vides}"
        }
    return {"statut": "OK", "message": "Aucune colonne entièrement vide"}


# ════════════════════════════════════════════════════════════
#  CONTRÔLE 5 — VALIDITÉ
#  Vérifie que les colonnes correspondent aux types attendus.
#  Tu passes un dictionnaire : {"nom_colonne": type_attendu}
#  Types Python acceptés : int, float, str, bool
#  Pour les dates, utiliser str (Teradata les retourne souvent
#  en str avant conversion explicite).
# ════════════════════════════════════════════════════════════
def check_validite(df: pd.DataFrame, types_attendus: dict) -> dict:
    erreurs = []
    for col, type_attendu in types_attendus.items():
        if col not in df.columns:
            erreurs.append(f"'{col}' absente")
            continue
        # Vérifier le type pandas équivalent
        col_data = df[col].dropna()
        if col_data.empty:
            continue
        if type_attendu in (int, float):
            if not pd.api.types.is_numeric_dtype(df[col]):
                erreurs.append(f"'{col}' : type attendu numérique, trouvé {df[col].dtype}")
        elif type_attendu == str:
            if not pd.api.types.is_string_dtype(df[col]) and not pd.api.types.is_object_dtype(df[col]):
                erreurs.append(f"'{col}' : type attendu texte, trouvé {df[col].dtype}")
        elif type_attendu == bool:
            if not pd.api.types.is_bool_dtype(df[col]):
                erreurs.append(f"'{col}' : type attendu booléen, trouvé {df[col].dtype}")

    if erreurs:
        return {"statut": "ECHEC", "message": " | ".join(erreurs)}
    return {"statut": "OK", "message": "Tous les types sont conformes"}


print("✅ Les 5 contrôles DQ définis :")
print("   check_unicite() | check_completude() | check_fraicheur()")
print("   check_coherence() | check_validite()")


## 2. Fonction principale — run_dq()

> Un seul appel pour lancer les 5 contrôles sur n'importe quel DataFrame.
> Tu configures les paramètres selon ta requête, elle fait le reste.


In [ ]:
def run_dq(df: pd.DataFrame,
            nom: str,
            min_lignes: int,
            max_lignes: int,
            col_date: str,
            date_debut: date,
            date_fin: date,
            types_attendus: dict) -> dict:
    """
    Lance les 5 contrôles DQ sur un DataFrame extrait de Teradata.

    Args:
        df             : DataFrame extrait
        nom            : Nom de la requête / extraction (pour les logs)
        min_lignes     : Nombre minimum de lignes attendu
        max_lignes     : Nombre maximum de lignes attendu
        col_date       : Nom de la colonne de date à vérifier
        date_debut     : Début de la période d'extraction attendue
        date_fin       : Fin de la période d'extraction attendue
        types_attendus : Dict {"colonne": type} ex: {"AMOUNT": float, "TRN_REF": str}

    Returns:
        dict avec statut global (valide/non valide) et détail par contrôle
    """
    t0 = time.time()

    logger.info("=" * 50)
    logger.info(f"DQ — {nom.upper()} | {len(df):,} lignes | {df.shape[1]} colonnes")
    logger.info("=" * 50)

    # ── Lancement des 5 contrôles ─────────────────────────────────────────────
    controles = {
        "Unicité"    : check_unicite(df),
        "Complétude" : check_completude(df, min_lignes, max_lignes),
        "Fraîcheur"  : check_fraicheur(df, col_date, date_debut, date_fin),
        "Cohérence"  : check_coherence(df),
        "Validité"   : check_validite(df, types_attendus),
    }

    # ── Affichage et logging ──────────────────────────────────────────────────
    all_ok = True
    print(f"\n{'─'*50}")
    print(f"  DQ — {nom}")
    print(f"{'─'*50}")
    for nom_ctrl, res in controles.items():
        icone = "✅" if res["statut"] == "OK" else "❌"
        print(f"  {icone} {nom_ctrl:<14} {res['message']}")
        if res["statut"] == "OK":
            logger.info(f"[{nom_ctrl}] OK — {res['message']}")
        else:
            logger.error(f"[{nom_ctrl}] ECHEC — {res['message']}")
            all_ok = False

    duree = round(time.time() - t0, 2)
    verdict = "✅ VALIDE" if all_ok else "❌ NON VALIDE"
    print(f"{'─'*50}")
    print(f"  Résultat : {verdict} | Durée : {duree}s")
    print(f"{'─'*50}\n")
    logger.info(f"Résultat global : {verdict} — {duree}s")

    return {
        "nom"       : nom,
        "timestamp" : datetime.now().isoformat(),
        "nb_lignes" : len(df),
        "valide"    : all_ok,
        "controles" : controles,
        "duree_s"   : duree
    }

print("✅ Fonction run_dq() définie")


## 3. Journalisation — log()

In [ ]:
def log(nom: str, rapport: dict, frequence: str,
        date_debut: date, date_fin: date) -> None:
    """
    Enregistre le résultat d'une exécution dans le journal CSV cumulatif.
    Une nouvelle ligne est ajoutée à chaque appel.
    """
    detail_erreurs = " | ".join([
        f"{k}: {v['message']}"
        for k, v in rapport["controles"].items()
        if v["statut"] == "ECHEC"
    ])

    record = {
        "Execution_Date" : rapport["timestamp"],
        "User"           : USER,
        "Extraction"     : nom,
        "Frequency"      : frequence,
        "Data_Begin"     : str(date_debut),
        "Data_End"       : str(date_fin),
        "Row_Number"     : rapport["nb_lignes"],
        "Duration(s)"    : rapport["duree_s"],
        "Status"         : "Good" if rapport["valide"] else "Bad",
        "Error_Reason"   : detail_erreurs
    }

    df_new    = pd.DataFrame([record])
    full_path = os.path.join(LOG_DIR, LOG_FILE_NAME)

    if os.path.isfile(full_path):
        df_old = pd.read_csv(full_path, sep=";", encoding="utf-8")
        df_new = pd.concat([df_old, df_new], ignore_index=True)

    df_new.to_csv(full_path, sep=";", index=False, encoding="utf-8")

print("✅ Fonction log() définie")


## 4. Données de démonstration

> Simule 3 extractions Teradata avec des structures différentes.
> En production, remplacer par `extraire(QUERY, "nom")`.


In [ ]:
hier = datetime.now().date() - timedelta(days=1)

# ── Extraction 1 : Transactions journalières ──────────────────────────────────
np.random.seed(42)
n1 = 250
df_transactions = pd.DataFrame({
    "TRN_REF"          : [f"TRN{str(i).zfill(8)}" for i in range(n1)],
    "ACCOUNT_ID"       : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n1)],
    "AMOUNT"           : np.random.lognormal(11, 2, n1).round(2),
    "CURRENCY"         : np.random.choice(["XOF","EUR","USD"], n1),
    "BOOKING_DATE"     : [hier] * n1,
    "TRANSACTION_TYPE" : np.random.choice(["VIR","PRE","CHQ"], n1),
})
# Injection d'anomalie : 2 doublons
df_transactions = pd.concat([df_transactions, df_transactions.iloc[[0,1]]], ignore_index=True)

# ── Extraction 2 : Comptes clients ───────────────────────────────────────────
n2 = 180
df_comptes = pd.DataFrame({
    "ACCOUNT_ID"   : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n2)],
    "CLIENT_NAME"  : [f"Client_{i}" for i in range(n2)],
    "ACCOUNT_TYPE" : np.random.choice(["CC","EP","CS","PRO"], n2),
    "OPEN_DATE"    : [hier - timedelta(days=np.random.randint(1, 3650))] * n2,
    "BALANCE"      : np.random.lognormal(13, 2, n2).round(2),
    "STATUS"       : np.random.choice(["ACTIVE","INACTIVE"], n2),
})
# Injection d'anomalie : colonne BRANCH_CODE entièrement vide
df_comptes["BRANCH_CODE"] = np.nan

# ── Extraction 3 : Mouvements comptables ─────────────────────────────────────
n3 = 400
df_mouvements = pd.DataFrame({
    "MVT_ID"       : range(n3),
    "ACCOUNT_ID"   : [f"SGC{np.random.randint(10000,99999)}" for _ in range(n3)],
    "DEBIT"        : np.random.lognormal(10, 2, n3).round(2),
    "CREDIT"       : np.random.lognormal(10, 2, n3).round(2),
    "MVT_DATE"     : [hier] * n3,
    "LABEL"        : [f"MVT_{i}" for i in range(n3)],
})
# Injection d'anomalie : MVT_ID stocké comme float au lieu d'int
df_mouvements["MVT_ID"] = df_mouvements["MVT_ID"].astype(float)

print("✅ 3 DataFrames de démonstration créés :")
print(f"   df_transactions : {len(df_transactions):,} lignes | {df_transactions.shape[1]} colonnes")
print(f"   df_comptes      : {len(df_comptes):,} lignes | {df_comptes.shape[1]} colonnes")
print(f"   df_mouvements   : {len(df_mouvements):,} lignes | {df_mouvements.shape[1]} colonnes")


## 5. Exemple avec 3 requêtes

> C'est ici que tout s'assemble. Chaque requête passe par `run_dq()` puis `log()`.
> Tu adaptes simplement les paramètres selon la requête.


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 1 — Transactions journalières
#  Attendu : entre 50 et 1000 lignes par jour
#  Colonne de date : BOOKING_DATE
#  Types : TRN_REF=str, AMOUNT=float, CURRENCY=str
# ════════════════════════════════════════════════════════════

# En production :
# df_transactions = extraire("""
#     SELECT TRN_REF, ACCOUNT_ID, AMOUNT, CURRENCY, BOOKING_DATE, TRANSACTION_TYPE
#     FROM SGCI_DB.TRANSACTIONS
#     WHERE BOOKING_DATE = CURRENT_DATE - 1
# """, "transactions_J1")

rapport_1 = run_dq(
    df             = df_transactions,
    nom            = "Transactions journalières",
    min_lignes     = 50,
    max_lignes     = 1000,
    col_date       = "BOOKING_DATE",
    date_debut     = hier,
    date_fin       = hier,
    types_attendus = {
        "TRN_REF"          : str,
        "ACCOUNT_ID"       : str,
        "AMOUNT"           : float,
        "CURRENCY"         : str,
        "TRANSACTION_TYPE" : str,
    }
)

log("transactions_journalieres", rapport_1, "Quotidien", hier, hier)


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 2 — Comptes clients
#  Attendu : entre 100 et 500 lignes (portefeuille stable)
#  Colonne de date : OPEN_DATE (date d'ouverture)
#  Types : ACCOUNT_ID=str, BALANCE=float, STATUS=str
# ════════════════════════════════════════════════════════════

# En production :
# df_comptes = extraire("""
#     SELECT ACCOUNT_ID, CLIENT_NAME, ACCOUNT_TYPE,
#            OPEN_DATE, BALANCE, STATUS, BRANCH_CODE
#     FROM SGCI_DB.ACCOUNTS
#     WHERE STATUS = 'ACTIVE'
# """, "comptes_clients")

rapport_2 = run_dq(
    df             = df_comptes,
    nom            = "Comptes clients actifs",
    min_lignes     = 100,
    max_lignes     = 500,
    col_date       = "OPEN_DATE",
    date_debut     = date(2015, 1, 1),  # comptes ouverts depuis 2015
    date_fin       = date.today(),
    types_attendus = {
        "ACCOUNT_ID"   : str,
        "BALANCE"      : float,
        "STATUS"       : str,
        "ACCOUNT_TYPE" : str,
    }
)

log("comptes_clients", rapport_2, "Quotidien", date(2015, 1, 1), date.today())


In [ ]:
# ════════════════════════════════════════════════════════════
#  REQUÊTE 3 — Mouvements comptables
#  Attendu : entre 200 et 2000 lignes par jour
#  Colonne de date : MVT_DATE
#  Types : MVT_ID=int, DEBIT=float, CREDIT=float, LABEL=str
# ════════════════════════════════════════════════════════════

# En production :
# df_mouvements = extraire("""
#     SELECT MVT_ID, ACCOUNT_ID, DEBIT, CREDIT, MVT_DATE, LABEL
#     FROM SGCI_DB.MOUVEMENTS_COMPTABLES
#     WHERE MVT_DATE = CURRENT_DATE - 1
# """, "mouvements_comptables")

rapport_3 = run_dq(
    df             = df_mouvements,
    nom            = "Mouvements comptables",
    min_lignes     = 200,
    max_lignes     = 2000,
    col_date       = "MVT_DATE",
    date_debut     = hier,
    date_fin       = hier,
    types_attendus = {
        "MVT_ID"     : int,    # ← devrait être int, sera float ici → ECHEC attendu
        "ACCOUNT_ID" : str,
        "DEBIT"      : float,
        "CREDIT"     : float,
        "LABEL"      : str,
    }
)

log("mouvements_comptables", rapport_3, "Quotidien", hier, hier)


## 6. Consultation du journal

In [ ]:
# Lire et afficher le journal complet
df_journal = pd.read_csv(os.path.join(LOG_DIR, LOG_FILE_NAME), sep=";")

print(f"📋 Journal — {len(df_journal)} exécution(s) enregistrée(s)\n")
df_journal


In [ ]:
# Résumé visuel : statut de chaque requête
print("=" * 55)
print("  BILAN DES 3 REQUÊTES")
print("=" * 55)
for rapport in [rapport_1, rapport_2, rapport_3]:
    icone = "✅" if rapport["valide"] else "❌"
    print(f"  {icone} {rapport['nom']:<35} {rapport['nb_lignes']:>5} lignes")
print("=" * 55)


---
## 📌 Comment utiliser ce framework pour une nouvelle requête

**3 étapes seulement :**

**1. Extraire les données**
```python
df = extraire("SELECT ... FROM SGCI_DB.MA_TABLE WHERE ...", "nom_extraction")
```

**2. Lancer les contrôles DQ**
```python
rapport = run_dq(
    df             = df,
    nom            = "nom_extraction",
    min_lignes     = 100,        # ← à adapter selon ta requête
    max_lignes     = 5000,       # ← à adapter selon ta requête
    col_date       = "MA_DATE",  # ← colonne de date dans ta table
    date_debut     = hier,       # ← début de la période attendue
    date_fin       = hier,       # ← fin de la période attendue
    types_attendus = {           # ← types de tes colonnes clés
        "MA_CLE"    : str,
        "MON_MONTANT" : float,
    }
)
```

**3. Journaliser**
```python
log("nom_extraction", rapport, "Quotidien", hier, hier)
```

---
*SGCI Data Engineering · Framework DQ Simplifié v1.0 · Juin 2026*
